In [0]:
%sql
-- look at results --
SELECT DATE(timestamp) AS day,
    COUNT(*) AS flagged_half_hours, 
    ROUND(MIN(anomaly_score), 3) AS worst_if_score
FROM elec_anomalies
WHERE is_anomaly
GROUP BY DATE(timestamp)
ORDER BY flagged_half_hours DESC;

- PATTERN ABOVE IN RESULTS: It’s mostly flagging winter Sundays, not real events.

### The results and what they show about the model bias

- MSTL split the `nd` into trend + daily pattern + weekly pattern + **residual**.
    - residual: shows what demand did beyond the normal patterns.
- The model was given **all 11 columns**, not the residual which is the primary feature for anomaly detection.
    - That included `weekday`, `month`, `nd` and `trend`, which still carry the patterns MSTL had removed.

#### SOURCE OF MODEL BIAS:
- Isolation forest picks columns **at random** - using sampling with replacement to create trees and entropy scores for measuring sample impurity anomaly detection; the residual was only used in about 1 in 11 splits amongst the number of trees.
- It flags values at the **edges** of a column. Sunday (6), Monday (0), January (1) and
  December (12) sit at the edges, so it flagged winter Sundays, not real events.
- Result: Christmas Day and the Covid lockdown had **0 flags**.

** To Fix:** train on the residual only, so the model sees only the surprise.